# Information Extraction Results


In [ ]:
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

INPUT_FILE = Path("results_verification.xlsm")

REPRESENTATIONS = {
    "full_text": "Full document",
    "section_based": "Operative section",
    "sentence_based": "Sentence-based",
}

FEATURES = [
    "Recipient",
    "Authority",
    "Legal action",
    "Legal object",
]


In [2]:
comparisons = pd.read_excel(INPUT_FILE, sheet_name="All Comparisons")
reviews = pd.read_excel(INPUT_FILE, sheet_name="Review Queue")

review_by_id = reviews.set_index("evaluation_id")
scored = comparisons.copy()

scored["TP"] = pd.to_numeric(scored["auto_TP"], errors="coerce").fillna(0)
scored["FP"] = pd.to_numeric(scored["auto_FP"], errors="coerce").fillna(0)
scored["FN"] = pd.to_numeric(scored["auto_FN"], errors="coerce").fillna(0)
scored["excluded"] = False

for i, row in scored.iterrows():
    evaluation_id = row["evaluation_id"]
    if evaluation_id not in review_by_id.index:
        continue

    review = review_by_id.loc[evaluation_id]
    excluded = str(review.get("exclude", "")).strip().lower() == "yes"
    scored.at[i, "excluded"] = excluded

    manual = [review.get("manual_TP"), review.get("manual_FP"), review.get("manual_FN")]
    if not excluded and all(pd.notna(v) and str(v).strip() != "" for v in manual):
        scored.at[i, "TP"] = float(manual[0])
        scored.at[i, "FP"] = float(manual[1])
        scored.at[i, "FN"] = float(manual[2])

scored = scored[~scored["excluded"]].copy()

availability = (
    comparisons.groupby(["document_id", "input_representation"])["input_available"]
    .max()
    .unstack(fill_value=0)
)

common_documents = set(
    availability[
        availability[list(REPRESENTATIONS)].eq(1).all(axis=1)
    ].index
)

def metrics(df):
    tp, fp, fn = df[["TP", "FP", "FN"]].sum()
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1

rows = []

for representation, label in REPRESENTATIONS.items():
    representation_rows = scored[scored["input_representation"] == representation]

    for feature in FEATURES + ["Overall"]:
        subset = representation_rows
        if feature != "Overall":
            subset = subset[subset["target"] == feature]

        common = subset[subset["document_id"].isin(common_documents)]

        rows.append([
            label,
            feature,
            *metrics(subset),
            *metrics(common),
        ])

results_table = pd.DataFrame(
    rows,
    columns=[
        "Input", "Feature",
        "All P", "All R", "All F1",
        "Common P", "Common R", "Common F1",
    ],
)

results_table.iloc[:, 2:] = results_table.iloc[:, 2:].round(2)


C:\Users\Gebruiker\AppData\Roaming\Python\Python314\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\Gebruiker\AppData\Roaming\Python\Python314\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [3]:
results_table


,Input,Feature,All P,All R,All F1,Common P,Common R,Common F1
0,Full document,Recipient,0.90,0.97,0.94,0.95,0.97,0.96
1,Full document,Authority,0.96,0.98,0.97,0.96,0.97,0.96
2,Full document,Legal action,0.99,1.00,0.99,0.99,1.00,0.99
3,Full document,Legal object,0.99,1.00,0.99,0.99,1.00,0.99
4,Full document,Overall,0.96,0.99,0.97,0.97,0.98,0.98
5,Operative section,Recipient,0.92,0.66,0.77,0.92,0.72,0.81
6,Operative section,Authority,0.86,0.47,0.61,0.86,0.59,0.70
7,Operative section,Legal action,0.99,0.79,0.88,0.99,0.99,0.99
8,Operative section,Legal object,0.99,0.79,0.88,0.99,0.99,0.99
9,Operative section,Overall,0.95,0.68,0.79,0.95,0.83,0.88
